<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzFit_User_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THzFit User Workflow

Code-cell-driven measured THz-TDS fitting for arbitrary isotropic multilayer samples.

## 1. Install And Import

Run the next cell first. In Colab it installs the package from GitHub and then prints the active package and dependency versions.

If you want to test a non-`main` branch in Colab before it is merged, set `THZSIM_GITHUB_BRANCH` or `THZSIM_PIP_SPEC` before running the install cell.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
IMPORT_EXCEPTION = None
NOTEBOOK_REPO_PATH = 'notebooks/THzFit_User_Workflow.ipynb'

def infer_colab_branch_from_url():
    if not IN_COLAB:
        return None
    try:
        from google.colab import output
        href = str(output.eval_js("window.location.href"))
    except Exception:
        return None
    marker = '/github/Podrimate/THz_sim_application/blob/'
    suffix = f'/{NOTEBOOK_REPO_PATH}'
    if marker not in href or suffix not in href:
        return None
    branch = href.split(marker, 1)[1].rsplit(suffix, 1)[0]
    return branch or None

DEFAULT_GITHUB_BRANCH = os.environ.get('THZSIM_GITHUB_BRANCH', infer_colab_branch_from_url() or 'main').strip() or 'main'
DEFAULT_PIP_SPEC = os.environ.get('THZSIM_PIP_SPEC', '').strip() or (
    f'https://github.com/Podrimate/THz_sim_application/archive/refs/heads/{DEFAULT_GITHUB_BRANCH}.zip'
)

def try_import_runtime_dependencies():
    global IMPORT_EXCEPTION
    try:
        import numpy  # noqa: F401
        import scipy  # noqa: F401
        import matplotlib  # noqa: F401
        return True
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return False

def try_import_thzsim2():
    global IMPORT_EXCEPTION
    try:
        importlib.invalidate_caches()
        import thzsim2  # noqa: F401
        return thzsim2
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return None

if IN_COLAB:
    pip_spec = DEFAULT_PIP_SPEC
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'thzsim2'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        '--no-deps',
        pip_spec,
    ])
    for module_name in [name for name in list(sys.modules) if name == 'thzsim2' or name.startswith('thzsim2.')]:
        sys.modules.pop(module_name, None)

if not try_import_runtime_dependencies():
    raise RuntimeError(
        'The Python runtime dependencies are currently broken. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime, then rerun the first cell. '
        "The notebook now installs only thzsim2 and leaves Colab's NumPy/SciPy/Matplotlib in place."
    )

thzsim2 = try_import_thzsim2()
PACKAGE_IMPORT_OK = thzsim2 is not None

if not PACKAGE_IMPORT_OK:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            thzsim2 = try_import_thzsim2()
            if thzsim2 is not None:
                PACKAGE_IMPORT_OK = True
                break

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime and rerun the first cell; '
        'locally open the notebook inside the repo.'
    )

import matplotlib
import numpy
import scipy

print(f'Install source spec: {DEFAULT_PIP_SPEC}')
print(f'Using thzsim2 from: {thzsim2.__file__}')
print(f'thzsim2 version: {getattr(thzsim2, "__version__", "unknown")}')
print(f'numpy version: {numpy.__version__}')
print(f'scipy version: {scipy.__version__}')
print(f'matplotlib version: {matplotlib.__version__}')
if IN_COLAB and DEFAULT_GITHUB_BRANCH == 'main' and 'THZSIM_PIP_SPEC' not in os.environ:
    print('Tip: set THZSIM_GITHUB_BRANCH or THZSIM_PIP_SPEC before rerunning this cell if you want to test a non-main branch.')


In [ ]:
from copy import deepcopy
from pathlib import Path
from pprint import pprint
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from thzsim2.io.trace_csv import read_trace_csv
from thzsim2.notebook_api import (
    Measurement,
    create_run_output_dir,
    enable_inline_plots,
    ensure_builtin_material_file,
    fit_param,
    inspect_trace_input,
    layers_from_definition,
    plot_objective_weighting,
    plot_trace_pair_preview,
    plot_trace_preview,
    prepare_reference,
    prepare_trace_pair_for_fit,
    preview_sample_response,
    run_measured_fit,
    save_fit_setup_snapshot,
    trace_spectrum,
    write_python_snapshot,
)

IN_COLAB = 'google.colab' in sys.modules
INLINE_PLOTS_ENABLED = enable_inline_plots()

def find_repo_path(relative_path):
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate_path = candidate / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    return None

def upload_single_csv(target_dir, *, prompt):
    print(prompt)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = Path(target_dir) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return path.resolve()

def print_mapping(title, mapping):
    display(Markdown(f'**{title}**'))
    for key, value in mapping.items():
        print(f'- {key}: {value}')

workflow_root = Path.cwd() / 'thz_fit_workflow_outputs'
workflow_root.mkdir(parents=True, exist_ok=True)
LOCAL_TEST_DATA = find_repo_path('Test_data_for_fitter')
print(f'Inline matplotlib enabled: {INLINE_PLOTS_ENABLED}')


## 2. Run Setup, Upload The Reference/Sample Files, And Preview The Data

Edit the next cell and rerun it until the previews look right.

What the main parameters mean:
- `run_name`: every saved artifact goes into a timestamped folder that includes this name.
- `measurement_mode`: choose `transmission` or `reflection` from the start so every later plot and save path matches the intended experiment.
- `use_colab_upload`: when `True`, Colab opens the file picker from your own computer.
- `use_local_example_if_available`: useful for local testing outside Colab with the bundled fitter datasets.
- `baseline_mode` and `crop_mode`: these control the preparation used for fitting. The plots below always let you inspect the result before fitting.
- `show_fft_preview`: when `True`, the notebook shows stacked FFT amplitude and FFT phase previews.
- `fft_floor_db`: clamps the plotted FFT floor so the preview does not dive to meaningless values like `-200 dB`.


In [ ]:
run_name = 'measured_drude_fit'
measurement_mode = 'transmission'  # 'transmission' or 'reflection'

use_colab_upload = IN_COLAB
use_local_example_if_available = not IN_COLAB
example_dataset = 'A11008858_transmission'
example_sample_name = 'SAMPLE.csv'

reference_csv_path = ''
sample_csv_path = ''
reference_time_column = None
reference_signal_column = None
sample_time_column = None
sample_signal_column = None

baseline_mode = 'auto_pre_pulse'  # safe default for most measured traces
baseline_window_samples = 50
crop_mode = 'auto'  # 'auto', 'manual', or 'none'
manual_crop_window_ps = None  # example: (2605.0, 2645.0)

show_fft_preview = True
fft_manual_limits_thz = None  # example: (0.0, 4.0)
fft_floor_db = -90.0


In [ ]:
run_dir = create_run_output_dir(run_name, root=workflow_root)
uploads_dir = run_dir / 'uploads'
python_dir = run_dir / 'python_snapshots'
config_dir = run_dir / 'configs'
for directory in (uploads_dir, python_dir, config_dir):
    directory.mkdir(parents=True, exist_ok=True)

selected_reference_csv = Path(reference_csv_path).expanduser().resolve() if reference_csv_path else None
selected_sample_csv = Path(sample_csv_path).expanduser().resolve() if sample_csv_path else None

if use_colab_upload:
    selected_reference_csv = upload_single_csv(uploads_dir, prompt='Upload the measured reference CSV.')
    selected_sample_csv = upload_single_csv(uploads_dir, prompt='Upload the measured sample CSV.')
elif use_local_example_if_available and LOCAL_TEST_DATA is not None:
    selected_reference_csv = (LOCAL_TEST_DATA / example_dataset / 'REFERENCE.csv').resolve()
    selected_sample_csv = (LOCAL_TEST_DATA / example_dataset / example_sample_name).resolve()

if selected_reference_csv is None or selected_sample_csv is None:
    raise RuntimeError('Please provide both reference and sample CSV files before running this cell.')

print(f'Run directory: {run_dir}')
print(f'Reference CSV: {selected_reference_csv}')
print(f'Sample CSV: {selected_sample_csv}')

reference_info = inspect_trace_input(
    selected_reference_csv,
    time_column=reference_time_column,
    signal_column=reference_signal_column,
)
sample_info = inspect_trace_input(
    selected_sample_csv,
    time_column=sample_time_column,
    signal_column=sample_signal_column,
)

plot_trace_preview(
    reference_info,
    title_prefix='Reference',
    show_fft=show_fft_preview,
    freq_limits_thz=fft_manual_limits_thz,
    fft_floor_db=fft_floor_db,
)
plot_trace_preview(
    sample_info,
    title_prefix='Sample',
    show_fft=show_fft_preview,
    freq_limits_thz=fft_manual_limits_thz,
    fft_floor_db=fft_floor_db,
)
print_mapping('Reference Summary', reference_info['summary'])
print_mapping('Sample Summary', sample_info['summary'])

prepared_traces = prepare_trace_pair_for_fit(
    selected_reference_csv,
    selected_sample_csv,
    reference_time_column=reference_time_column,
    reference_signal_column=reference_signal_column,
    sample_time_column=sample_time_column,
    sample_signal_column=sample_signal_column,
    baseline_mode=baseline_mode,
    baseline_window_samples=baseline_window_samples,
    crop_mode=crop_mode,
    crop_time_window_ps=manual_crop_window_ps if crop_mode == 'manual' else None,
)
plot_trace_pair_preview(prepared_traces)
print_mapping('Prepared Pair Summary', prepared_traces.metadata)
write_python_snapshot(
    python_dir / 'part2_trace_selection.py',
    run_name=run_name,
    measurement_mode=measurement_mode,
    selected_reference_csv=str(selected_reference_csv),
    selected_sample_csv=str(selected_sample_csv),
    baseline_mode=baseline_mode,
    baseline_window_samples=baseline_window_samples,
    crop_mode=crop_mode,
    manual_crop_window_ps=manual_crop_window_ps,
    show_fft_preview=show_fft_preview,
    fft_manual_limits_thz=fft_manual_limits_thz,
    fft_floor_db=fft_floor_db,
)


## 3. Measurement Settings

Edit the next cell only if you need something more advanced than the simplest near-normal fit.

What the parameters mean:
- `angle_guess_deg`: the incident angle used by the forward model.
- `fit_angle_deg=True`: turns `angle_deg` into a fit parameter by wrapping it in `fit_param(...)`.
- `polarization`: choose `s`, `p`, or `mixed`.
- `fit_polarization_mix=True`: only meaningful when `polarization='mixed'`; it fits the continuous `s/p` mixture between `0` and `1`.
- `reference_standard_kind`: use `identity` unless you intentionally want to normalize by a modeled stack.

Important note:
- the notebook can fit `angle_deg`
- it cannot fit the discrete choice between `s` and `p` directly
- if the polarization is uncertain, use `polarization='mixed'` and fit `polarization_mix`


In [ ]:
fit_angle_deg = False
angle_guess_deg = 0.0
angle_fit_bounds_deg = (0.0, 30.0)

polarization = 's'  # 's', 'p', or 'mixed'
fit_polarization_mix = False
polarization_mix_guess = 0.5
polarization_mix_fit_bounds = (0.0, 1.0)

reference_standard_kind = 'identity'

if fit_polarization_mix and polarization != 'mixed':
    raise ValueError("Set polarization='mixed' if you want to fit polarization_mix.")

angle_deg = (
    fit_param(angle_guess_deg, angle_fit_bounds_deg[0], angle_fit_bounds_deg[1], label='measurement_angle_deg')
    if fit_angle_deg
    else float(angle_guess_deg)
)
polarization_mix = None
if polarization == 'mixed':
    polarization_mix = (
        fit_param(
            polarization_mix_guess,
            polarization_mix_fit_bounds[0],
            polarization_mix_fit_bounds[1],
            label='measurement_polarization_mix',
        )
        if fit_polarization_mix
        else float(polarization_mix_guess)
    )

measurement = Measurement(
    mode=measurement_mode,
    angle_deg=angle_deg,
    polarization=polarization,
    polarization_mix=polarization_mix,
    reference_standard={'kind': reference_standard_kind},
)
print(measurement)
print("Tip: use fit_angle_deg=True to recover angle_deg from the data.")
print("Tip: use polarization='mixed' plus fit_polarization_mix=True when the actual s/p mix is unknown.")


## 4. Sample (Stack) Definition

The next code cell is the main user-editable sample definition.

How to read it:
- each list entry is one layer in propagation order
- `thickness_um` is always in micrometers
- `fit_param(...)` marks a parameter that the fitter is allowed to move
- constant values stay fixed
- you can build any isotropic multilayer by stacking more dictionaries

The cell also lets you set `direct_nk_thickness_guess_um`.
When the measurement is normal-incidence transmission, the preview cell uses that thickness guess to plot an approximate direct `n,k` estimate next to the model `n,k` curves.

The cell already contains three editable templates:
- `one_layer_drude`
- `two_layer_lorentz_plus_si`
- `three_layer_lorentz_plus_si_plus_sio2_backside`


In [ ]:
materials_dir = run_dir / 'materials'
si_nk_csv = ensure_builtin_material_file('si_thz_nk.csv', out_dir=materials_dir)
sio2_nk_csv = ensure_builtin_material_file('sio2_lossy_thz_nk.csv', out_dir=materials_dir)

sample_template_name = 'one_layer_drude'
direct_nk_thickness_guess_um = 550.0

sample_templates = {
    'one_layer_drude': [
        {
            'name': 'drude_film',
            'thickness_um': fit_param(550.0, 200.0, 900.0, label='film_thickness_um'),
            'material': {
                'kind': 'Drude',
                'eps_inf': fit_param(11.0, 4.0, 20.0, label='film_eps_inf'),
                'plasma_freq_thz': fit_param(1.1, 0.01, 5.0, label='film_plasma_freq_thz'),
                'gamma_thz': fit_param(0.08, 0.001, 2.0, label='film_gamma_thz'),
            },
        }
    ],
    'two_layer_lorentz_plus_si': [
        {
            'name': 'lorentz_coating',
            'thickness_um': fit_param(180.0, 60.0, 420.0, label='coating_thickness_um'),
            'material': {
                'kind': 'DrudeLorentz',
                'eps_inf': fit_param(3.4, 1.0, 8.0, label='coating_eps_inf'),
                'drude_plasma_freq_thz': 0.0,
                'drude_gamma_thz': 1.0,
                'oscillators': [
                    {
                        'delta_eps': fit_param(1.2, 0.1, 4.0, label='osc1_delta_eps'),
                        'resonance_thz': fit_param(0.8, 0.2, 1.6, label='osc1_resonance_thz'),
                        'gamma_thz': fit_param(0.12, 0.01, 0.6, label='osc1_gamma_thz'),
                    },
                    {
                        'delta_eps': fit_param(0.8, 0.1, 4.0, label='osc2_delta_eps'),
                        'resonance_thz': fit_param(1.7, 0.6, 2.8, label='osc2_resonance_thz'),
                        'gamma_thz': fit_param(0.2, 0.01, 0.8, label='osc2_gamma_thz'),
                    },
                ],
            },
        },
        {
            'name': 'si_layer',
            'thickness_um': fit_param(500.0, 200.0, 1000.0, label='si_thickness_um'),
            'material': {'kind': 'NKFile', 'path': str(si_nk_csv)},
        },
    ],
    'three_layer_lorentz_plus_si_plus_sio2_backside': [
        {
            'name': 'lorentz_coating',
            'thickness_um': fit_param(180.0, 60.0, 420.0, label='coating_thickness_um'),
            'material': {
                'kind': 'DrudeLorentz',
                'eps_inf': fit_param(3.4, 1.0, 8.0, label='coating_eps_inf'),
                'drude_plasma_freq_thz': 0.0,
                'drude_gamma_thz': 1.0,
                'oscillators': [
                    {
                        'delta_eps': fit_param(1.2, 0.1, 4.0, label='osc1_delta_eps'),
                        'resonance_thz': fit_param(0.8, 0.2, 1.6, label='osc1_resonance_thz'),
                        'gamma_thz': fit_param(0.12, 0.01, 0.6, label='osc1_gamma_thz'),
                    },
                    {
                        'delta_eps': fit_param(0.8, 0.1, 4.0, label='osc2_delta_eps'),
                        'resonance_thz': fit_param(1.7, 0.6, 2.8, label='osc2_resonance_thz'),
                        'gamma_thz': fit_param(0.2, 0.01, 0.8, label='osc2_gamma_thz'),
                    },
                ],
            },
        },
        {
            'name': 'si_layer',
            'thickness_um': fit_param(500.0, 200.0, 1000.0, label='si_thickness_um'),
            'material': {'kind': 'NKFile', 'path': str(si_nk_csv)},
        },
        {
            'name': 'sio2_backside',
            'thickness_um': 1.0e7,
            'material': {'kind': 'NKFile', 'path': str(sio2_nk_csv)},
        },
    ],
}

sample_n_in = 1.0
sample_n_out = 1.0
overlay_imported_nk = True
sample_definition = deepcopy(sample_templates[sample_template_name])
pprint(sample_definition)
print(f'Direct n,k thickness guess (um): {direct_nk_thickness_guess_um}')


## 5. Fit Controls, Sample Preview, And Saved Snapshots

The next cell does three things:
- builds the editable sample stack
- previews the simulated sample against the uploaded data before fitting
- saves reproducible setup snapshots before the fit runs

What to look at:
- `metric='weighted_data_fit'` plus `weighting=...` makes the fit focus more strongly on the pulse region instead of long flat tails
- `global_method='differential_evolution'` and `global_restarts` give the fit a stronger global search before local polishing
- the sample preview should let you judge whether the current stack is even in the right neighborhood before spending time on the fit
- `enable_delay_recovery_for_tests` is mainly for synthetic translation tests or badly time-shifted reflection uploads, not the normal measured-data transmission path


In [ ]:
metric = 'weighted_data_fit'
weighting = {
    'mode': 'trace_amplitude',
    'floor': 0.03,
    'power': 2.0,
    'smooth_window_samples': 31,
}
max_internal_reflections = 4
optimizer = {
    'global_method': 'differential_evolution',
    'global_restarts': 2,
    'method': 'L-BFGS-B',
    'options': {'maxiter': 220},
    'global_options': {
        'maxiter': 20,
        'popsize': 18,
        'seed': 11,
        'polish': False,
        'tol': 1e-7,
        'init': 'sobol',
    },
    'fd_rel_step': 1e-5,
}

enable_delay_recovery_for_tests = False
delay_options = None if not enable_delay_recovery_for_tests else {'enabled': True, 'search_window_ps': 120.0}

layers = layers_from_definition(sample_definition)
preview_reference_result = prepare_reference(
    prepared_traces.processed_reference,
    output_root=run_dir / 'preview_runs',
    run_label='fit-preview-reference',
)
sample_preview_result, sample_preview_simulation, _, _ = preview_sample_response(
    reference_result=preview_reference_result,
    layers=layers,
    out_dir=run_dir / 'preview_sample',
    observed_trace=prepared_traces.processed_sample,
    measurement=measurement,
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    max_internal_reflections=max_internal_reflections,
    show_fft=show_fft_preview,
    freq_limits_thz=fft_manual_limits_thz,
    thickness_guess_um=direct_nk_thickness_guess_um,
    fft_floor_db=fft_floor_db,
)
if direct_nk_thickness_guess_um is not None and sample_preview_simulation.get('nk_estimate') is None:
    print('Direct n,k estimate note: available for normal-incidence transmission previews only.')
if metric == 'weighted_data_fit':
    plot_objective_weighting(
        prepared_traces.processed_sample,
        weighting,
        title='Objective Weighting Preview',
    )

fit_setup_json_path = config_dir / 'fit_setup.json'
fit_definition_py_path = python_dir / 'sample_definition.py'

write_python_snapshot(
    fit_definition_py_path,
    measurement_mode=measurement_mode,
    angle_deg=angle_deg,
    polarization=polarization,
    polarization_mix=polarization_mix,
    fit_angle_deg=fit_angle_deg,
    fit_polarization_mix=fit_polarization_mix,
    sample_definition=sample_definition,
    metric=metric,
    weighting=weighting,
    max_internal_reflections=max_internal_reflections,
    optimizer=optimizer,
    delay_options=delay_options,
    direct_nk_thickness_guess_um=direct_nk_thickness_guess_um,
    sample_n_in=sample_n_in,
    sample_n_out=sample_n_out,
)
save_fit_setup_snapshot(
    fit_setup_json_path,
    reference_trace={'path': str(selected_reference_csv), 'time_column': reference_time_column, 'signal_column': reference_signal_column},
    sample_trace={'path': str(selected_sample_csv), 'time_column': sample_time_column, 'signal_column': sample_signal_column},
    layers=layers,
    measurement=measurement,
    preprocessing={
        'baseline_mode': baseline_mode,
        'baseline_window_samples': baseline_window_samples,
        'crop_mode': crop_mode,
        'crop_time_window_ps': manual_crop_window_ps if crop_mode == 'manual' else None,
    },
    optimizer=optimizer,
    metric=metric,
    max_internal_reflections=max_internal_reflections,
    delay_options=delay_options,
    weighting=weighting,
    out_dir=str(run_dir / 'fit_run'),
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    notes='Created from THzFit_User_Workflow.ipynb',
)
print(f'Saved machine-readable setup: {fit_setup_json_path}')
print(f'Saved human-readable snapshot: {fit_definition_py_path}')


## 6. Run The Fit And Review The Result

The next cell runs the fit and prints the key diagnostics.

What a good result should look like:
- the fitted trace overlaps the processed sample trace in the main pulse region
- `data_fit`, `weighted_data_fit`, `normalized_mse`, and `relative_l2` are much smaller than the initial values
- the residual becomes structure-light instead of containing a large shifted copy of the pulse
- the FFT amplitude and phase comparisons stay qualitatively consistent with the measured sample


In [ ]:
fit_result_bundle = run_measured_fit(
    prepared_traces,
    layers,
    out_dir=run_dir / 'fit_run',
    measurement=measurement,
    optimizer=optimizer,
    metric=metric,
    max_internal_reflections=max_internal_reflections,
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    delay_options=delay_options,
    weighting=weighting,
)

fit_result = fit_result_bundle.fit_result
print_mapping('Initial Residual Metrics', fit_result['initial_residual_metrics'])
print_mapping('Final Residual Metrics', fit_result['residual_metrics'])
print_mapping('Recovered Parameters', fit_result['recovered_parameters'])
print_mapping('Delay Recovery', fit_result['delay_recovery'])
print(f'Optimizer stage used: {fit_result["optimizer_stage"]}')
print(f'Measured fit outputs: {fit_result_bundle.out_dir}')

processed_sample = fit_result_bundle.prepared_traces.processed_sample
initial_trace = read_trace_csv(fit_result_bundle.artifact_paths['initial_sample_trace_csv'])
fitted_trace = read_trace_csv(fit_result_bundle.artifact_paths['fitted_sample_trace_csv'])
residual_trace = read_trace_csv(fit_result_bundle.artifact_paths['residual_trace_csv'])

if show_fft_preview:
    fig, axes = plt.subplots(4, 1, figsize=(11, 14), sharex=False)
else:
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
axes = np.atleast_1d(axes)

axes[0].plot(processed_sample.time_ps, processed_sample.trace, label='processed sample')
axes[0].plot(initial_trace.time_ps, initial_trace.trace, label='initial guess', alpha=0.8)
axes[0].plot(fitted_trace.time_ps, fitted_trace.trace, label='fit', linewidth=1.4)
axes[0].set_title('Measured Sample And Fit Overlay')
axes[0].set_xlabel('Time (ps)')
axes[0].set_ylabel('Signal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(residual_trace.time_ps, residual_trace.trace, label='residual')
axes[1].set_title('Residual Trace')
axes[1].set_xlabel('Time (ps)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

if show_fft_preview:
    freq_sample, amp_sample, phase_sample = trace_spectrum(processed_sample, floor_db=fft_floor_db)
    freq_initial, amp_initial, phase_initial = trace_spectrum(initial_trace, floor_db=fft_floor_db)
    freq_fit, amp_fit, phase_fit = trace_spectrum(fitted_trace, floor_db=fft_floor_db)

    axes[2].plot(freq_sample, amp_sample, label='processed sample')
    axes[2].plot(freq_initial, amp_initial, label='initial guess', alpha=0.8)
    axes[2].plot(freq_fit, amp_fit, label='fit')
    axes[2].set_title('FFT Amplitude Comparison (Relative)')
    axes[2].set_xlabel('Frequency (THz)')
    axes[2].set_ylabel('Amplitude (dB)')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    axes[3].plot(freq_sample, phase_sample, label='processed sample')
    axes[3].plot(freq_initial, phase_initial, label='initial guess', alpha=0.8)
    axes[3].plot(freq_fit, phase_fit, label='fit')
    axes[3].set_title('FFT Phase Comparison (Unwrapped)')
    axes[3].set_xlabel('Frequency (THz)')
    axes[3].set_ylabel('Phase (rad)')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()

    if fft_manual_limits_thz is not None:
        axes[2].set_xlim(float(fft_manual_limits_thz[0]), float(fft_manual_limits_thz[1]))
        axes[3].set_xlim(float(fft_manual_limits_thz[0]), float(fft_manual_limits_thz[1]))

fig.tight_layout()
display(fig)
plt.close(fig)
